# Notebook 11: 模型可解释性 -- SHAP & 特征分析 (山东 15min)
## Model Explainability: SHAP & Feature Analysis on Shandong 15-Minute Data

**目标**: 理解 SHAP 原理，用 TreeExplainer 解释 XGBoost 负荷预测模型的决策，
掌握 waterfall/summary/beeswarm 图，对比内置重要性 vs SHAP 重要性。

### SHAP 核心思想 / Core Idea
```
prediction = base_value + sum(shap_value_i)
```
- **base_value**: 模型在所有训练样本上的平均预测值
- **shap_value_i**: 第 i 个特征对这个样本的贡献（正=推高、负=拉低）

> SHAP 值比简单特征重要性更有信息量：不仅告诉你特征重不重要，
> 还告诉你在特定样本上特征推高了还是拉低了预测值。

In [ ]:
# -- 导入 / Imports --
import os, warnings, logging
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.WARNING)
pio.renderers.default = "notebook"

from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.forecaster import XGBoostForecaster
from ellectric.pipeline.shap_explainer import (
    explain_xgboost_sample, feature_importance_ranking)
from ellectric.config import TimeConfig

print("All imports OK / 所有导入成功")
print(f"  TimeConfig.points_per_day = {TimeConfig.points_per_day}")

---
## 步骤 1: 加载数据 + 特征工程 + 训练 XGBoost / Step 1: Load + Feature Engineering + Train

使用山东 2024-07~2024-08 数据，添加全部 3 层特征，训练 XGBoost。

In [ ]:
# -- 加载数据 + 清洗 --
loader = create_loader("shandong")
df = loader.load_data("2024-07-01", "2024-08-31")
df = clean_data(df)
print(f"Loaded / 已加载: {len(df)} rows, {TimeConfig.freq} ({TimeConfig.points_per_day} pts/day)")

# -- 特征工程 (Tier 1+2+3) --
df_load = df[["timestamp", "load_mw"]].copy()
fe = FeatureEngineer()
df_feat = fe.add_tier1_features(df_load)
df_feat = fe.add_tier2_features(df_feat)
df_feat = fe.add_tier3_features(df_feat)

feature_cols = [c for c in df_feat.columns if c not in ("timestamp", "load_mw")]
print(f"Features / 特征 ({len(feature_cols)}): {feature_cols}")

# 跳过 NaN 行
lag = TimeConfig.points_per_day
X = df_feat[feature_cols].iloc[lag:]
y = df_feat["load_mw"].iloc[lag:]

# -- 训练 XGBoost --
MODEL_DIR = "../models"; os.makedirs(MODEL_DIR, exist_ok=True)
xgb = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
result = xgb.train_evaluate(X, y, n_splits=3, gap=lag)
xgb.save_model(f"{MODEL_DIR}/xgboost_load_shandong.joblib")

m = result["metrics"]
print(f"XGBoost / 训练完成: MAE={m['mae']:.1f} MW, RMSE={m['rmse']:.1f} MW, "
      f"MAPE={m.get('mape',0)*100:.2f}%, R2={m.get('r2',0):.3f}")

---
## 步骤 2: SHAP Waterfall -- 单样本解释 / Step 2: Single-Sample Waterfall

选择一条测试样本，用 SHAP TreeExplainer 计算每个特征的贡献。
修改 `sample_idx` 可查看不同时段。

In [ ]:
# -- XGBoost SHAP Waterfall --
sample_idx = 0  # 修改此值查看不同样本
try:
    fig = explain_xgboost_sample(model=xgb, X=X, sample_idx=sample_idx, max_display=12)
    fig.show()
    print(f"Sample {sample_idx} / 样本: time={df_load['timestamp'].iloc[lag + sample_idx]}, "
          f"load={y.iloc[sample_idx]:.0f} MW")
except Exception as e:
    print(f"SHAP failed / SHAP 失败: {e}")
    print("  Run: pip install shap")

---
## 步骤 3: SHAP Summary Bar -- 全局特征重要性 / Step 3: Global SHAP Importance

在所有样本上平均 |SHAP 值|。与 XGBoost 内置 `feature_importances_` 互补：
- 内置: 基于分裂次数/信息增益
- SHAP: 基于特征对预测的实际影响（方向感知）

In [ ]:
# -- SHAP Summary Bar (Global) --
try:
    import shap as _shap
    feat_cols = xgb._feature_cols
    n_s = min(500, len(X))
    X_s = X[feat_cols].iloc[:n_s]
    print(f"Computing SHAP on {n_s} samples / 计算 {n_s} 样本...")

    explainer = _shap.TreeExplainer(xgb._model)
    sv = explainer.shap_values(X_s)
    mean_abs = np.abs(sv).mean(axis=0)
    sort_idx = np.argsort(mean_abs)[::-1]

    top_n = min(10, len(feat_cols))
    top_f = [feat_cols[i] for i in sort_idx[:top_n]]
    top_v = [float(mean_abs[i]) for i in sort_idx[:top_n]]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=top_v, y=top_f, orientation="h", marker_color="#2ca02c",
        text=[f"{v:.2f}" for v in top_v], textposition="outside"))
    fig.update_layout(
        title=f"XGBoost SHAP Summary — Top {top_n} Features (Shandong 15min)<br>"
              f"<sup>Based on {n_s} samples, mean |SHAP value|</sup>",
        xaxis_title="Mean |SHAP Value|",
        yaxis=dict(autorange="reversed"),
        height=380, margin=dict(l=180, r=60, t=80, b=40))
    fig.show()
except ImportError:
    print("shap not installed. Run: pip install shap")
except Exception as e:
    print(f"SHAP failed / SHAP 失败: {e}")

---
## 步骤 4: SHAP Beeswarm + 内置 vs SHAP 对比 / Step 4: Beeswarm + Built-in vs SHAP

Beeswarm 同时展示重要性和影响方向。然后对比两种重要性度量。

In [ ]:
# -- SHAP Beeswarm --
try:
    import shap as _shap
    import matplotlib.pyplot as plt
    n_s2 = min(300, len(X))
    X_s2 = X[feat_cols].iloc[:n_s2]
    explainer2 = _shap.TreeExplainer(xgb._model)
    sv2 = explainer2.shap_values(X_s2)

    top8 = [feat_cols[i] for i in sort_idx[:8]]
    X_top = X_s2[top8]
    sv_top = sv2[:, [list(feat_cols).index(f) for f in top8]]
    _shap.summary_plot(sv_top, X_top, show=False, max_display=8)
    plt.tight_layout(); plt.show()
except ImportError:
    print("shap or matplotlib not installed")
except Exception as e:
    print(f"Beeswarm failed: {e}")

# -- 内置重要性 vs SHAP --
try:
    builtin = xgb._model.feature_importances_
    shap_imp = mean_abs
    bn = builtin / builtin.max() if builtin.max() > 0 else builtin
    sn = shap_imp / shap_imp.max() if shap_imp.max() > 0 else shap_imp

    comp_data = [{"Feature": feat_cols[i], "Built-in (norm)": round(float(bn[i]),3),
        "SHAP (norm)": round(float(sn[i]),3)} for i in range(len(feat_cols))]
    comp_df = pd.DataFrame(comp_data).sort_values("SHAP (norm)", ascending=False)
    print("Comparison / 对比表 (sorted by SHAP):")
    display(comp_df)

    top_n2 = min(10, len(feat_cols))
    top_comp = comp_df.head(top_n2)
    fig = go.Figure()
    fig.add_trace(go.Bar(y=top_comp["Feature"], x=top_comp["Built-in (norm)"],
        name="Built-in / 内置", orientation="h", marker_color="#1f77b4"))
    fig.add_trace(go.Bar(y=top_comp["Feature"], x=top_comp["SHAP (norm)"],
        name="SHAP", orientation="h", marker_color="#ff7f0e"))
    fig.update_layout(title="Built-in vs SHAP Feature Importance (Normalized)",
        xaxis_title="Normalized Importance", yaxis=dict(autorange="reversed"),
        barmode="group", height=380, margin=dict(l=180, r=40, t=60, b=40))
    fig.show()
except NameError:
    print("SHAP values not computed. Run Step 3 first.")

---
## 步骤 5: 高峰/低谷时段对比 + 特征相关性 / Step 5: Peak vs Off-Peak + Correlation

对比高负荷和低负荷样本的 SHAP 解释差异。分析 15 分钟粒度下的特征相关性。

In [ ]:
# -- 高峰/低谷样本对照 / Peak vs Off-peak SHAP --
low_idx = y.sort_values().index[0]  # 最低负荷
high_idx = y.sort_values(ascending=False).index[0]  # 最高负荷
print(f"Low load / 低负荷: idx={low_idx}, {y.iloc[low_idx]:.0f} MW")
print(f"  Time: {df_load['timestamp'].iloc[lag + low_idx]}")
print(f"High load / 高峰负荷: idx={high_idx}, {y.iloc[high_idx]:.0f} MW")
print(f"  Time: {df_load['timestamp'].iloc[lag + high_idx]}")

for label, idx in [("Low Load", low_idx), ("High Load / Peak", high_idx)]:
    try:
        fig = explain_xgboost_sample(model=xgb, X=X, sample_idx=idx, max_display=10)
        fig.update_layout(title=f"SHAP Waterfall — {label} (Sample {idx})")
        fig.show()
    except Exception as e:
        print(f"  Failed: {e}")

# -- 特征相关性矩阵 / Feature Correlation --
import plotly.express as px
corr = X[feature_cols].corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, title="Feature Correlation Matrix / 特征相关性 — 15min Granularity")
fig.update_layout(height=500, width=680)
fig.show()

---
## 讨论与思考 / Discussion & Questions

### 15 分钟粒度对 SHAP 的影响 / Impact of 15-min Granularity

| 维度 | 小时级 (24/d) | 15 分钟级 (96/d) |
|------|-------------|----------------|
| lag 特征 | lag_24h 强相关 | lag_96 更强（更短间隔） |
| SHAP 计算 | 快 | 慢（4x 样本） |
| 解释粒度 | 小时级决策 | 15 分钟级精确决策 |

### 局限性 / Limitations
- 特征高度相关时 SHAP 值分配不稳定
- SHAP 反映的是模型学到的模式，不一定是物理因果关系
- 线性模型系数仅在标准化后有可比性

### 思考题 / Questions
1. 哪个特征 SHAP 贡献最大？它的方向是否符合物理直觉？
2. 高峰/低谷时段的特征贡献模式有何不同？
3. 内置重要性和 SHAP 重要性的排名一致吗？哪个更可靠？
4. 15 分钟粒度下 lag_96 vs rolling_mean_96 哪个更重要？为什么？

---
**学习笔记 / Key Takeaways**
1. SHAP TreeExplainer 成功解释了 XGBoost 负荷预测（96 点/日）
2. 单样本 waterfall + 全局 bar + beeswarm 提供了完整的可解释性视角
3. 15 分钟粒度特征相关性更高，SHAP 分配更需注意多重共线性